# Scaling computations using parallel computing with Dagger

In [1]:
# Add some workers with threads
using Distributed; addprocs(2; exeflags=["--project=$(joinpath(pwd(), ".."))", "--threads=2"])
@everywhere using Dagger
@everywhere using BenchmarkTools

## How to program with tasks

In [2]:
t = Dagger.@spawn 1+2
@show t
fetch(t)

t = EagerThunk (running)


3

This makes a Dagger task and executes it asynchronously. `fetch` gets the result once it's run.

In [3]:
a = Dagger.@spawn 1 + 1
b = Dagger.@spawn a * 2
c = Dagger.@spawn a / 2
d = Dagger.@spawn b - c
fetch(d)

3.0

`b` and `c` are run in parallel, as they don't depend on each other.

In [4]:
# We use @everywhere here to make sure all workers know about our `who_am_i` function
@everywhere function who_am_i()
    id = Distributed.myid() # Which worker are we? Starts from 1
    tid = Threads.threadid()
    println("Hello from thread $tid on worker $id")
end

wait.([Dagger.@spawn who_am_i() for i in 1:20]); # This might run locally or on any worker; try re-running this!

      From worker 3:	Hello from thread 2 on worker 3
      From worker 3:	Hello from thread 1 on worker 3
      From worker 2:	Hello from thread 1 on worker 2
      From worker 3:	Hello from thread 1 on worker 3
      From worker 2:	Hello from thread 2 on worker 2
      From worker 2:	Hello from thread 1 on worker 2
      From worker 3:	Hello from thread 2 on worker 3
      From worker 3:	Hello from thread 2 on worker 3
      From worker 2:	Hello from thread 2 on worker 2
      From worker 3:	Hello from thread 1 on worker 3
      From worker 2:	Hello from thread 1 on worker 2
      From worker 3:	Hello from thread 1 on worker 3
      From worker 3:	Hello from thread 1 on worker 3
      From worker 2:	Hello from thread 2 on worker 2
      From worker 2:	Hello from thread 1 on worker 2
      From worker 3:	Hello from thread 1 on worker 3
      From worker 2:	Hello from thread 2 on worker 2
      From worker 2:	Hello from thread 1 on worker 2
      From worker 2:	Hello from thread 1 on wo

Now do some more expensive computations:

In [5]:
# A function which does something expensive with arrays
@everywhere function some_function(i, arrs...)
    @nospecialize arrs
    total = 0.0
    for arr in arrs
        total += i * sum(inv(arr))
    end
    return rand(0.0:abs(total), 100, 100)
end

# A function which generates a large DAG of expensive computations
function create_dag(f, Ns...; use_dagger=false)
    last_nodes = []
    current_nodes = []
    
    push!(last_nodes, rand(100, 100))

    for N in Ns
        println("Generating $N nodes")
        for i in 1:N
            if use_dagger
                push!(current_nodes, Dagger.@spawn f(i, last_nodes...))
            else
                push!(current_nodes, f(i, last_nodes...))
            end
        end
        empty!(last_nodes)
        append!(last_nodes, current_nodes)
        empty!(current_nodes)
    end
    
    return fetch.(last_nodes)
end

# Disable BLAS multithreading - while BLAS multithreading is very efficient,
# many workloads do not automatically parallelize themselves :)
using LinearAlgebra; BLAS.set_num_threads(1)

In [6]:
# Let's try just running this code in serial:
@time create_dag(some_function, 100, 50; use_dagger=false);

Generating 100 nodes
Generating 50 nodes
  1.435390 seconds (606.36 k allocations: 693.027 MiB, 13.69% gc time, 31.24% compilation time: 29% of which was recompilation)


In [7]:
# Let's go faster with Dagger:
@time create_dag(some_function, 100, 50; use_dagger=true);

Generating 100 nodes
Generating 50 nodes
  1.777810 seconds (1.12 M allocations: 70.407 MiB, 0.59% gc time, 32.41% compilation time)


Doesn't run faster due to CPU bottlenecking: linear algebra doesn't scale linearly here.

## Data movement
Dagger can move data between workers or execute where the data already is; it chooses whichever is more efficient.

## Array programming with Dagger
Distributed arrays allow multithreading and multiprocessing with arrays.

## Tabular programming in Dagger
Distributed tables allow operations on large dataframes over multiple workers.